In [ ]:
import os
os.chdir('/home/cbn-gpu12/FNF/VLM/LLaVA/LLaVA-Med')


from peft import PeftModel
from huggingface_hub import create_repo

import warnings
warnings.filterwarnings("ignore")
import random
import torch
from torch.utils.data.dataset import Dataset
from torch.nn.utils.rnn import pad_sequence
from datasets import load_dataset
from io import BytesIO
import requests
from datetime import datetime
import gc
import json
import time 
from collections import defaultdict 
from transformers import Trainer, TrainingArguments
from peft import LoraConfig, LoraModel, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from torch.optim.lr_scheduler import ReduceLROnPlateau 
from llava.constants import DEFAULT_IMAGE_TOKEN, IMAGE_TOKEN_INDEX
from llava.conversation import Conversation
from llava.mm_utils import tokenizer_image_token, process_images
from llava.model.builder import load_pretrained_model
from llava.conversation import conv_templates
from tqdm import tqdm
from torch.utils.data import DataLoader
import glob
import torchvision.transforms as transforms
from torch.utils.data import Dataset
from concurrent.futures import ThreadPoolExecutor
from PIL import Image
from multiprocessing import Pool, cpu_count
import pydicom
import numpy as np
from concurrent.futures import ProcessPoolExecutor
import uuid
import pickle
from transformers import LlamaTokenizer
from llava.model import LlavaMistralForCausalLM  # LLaVA-Med의 모델 클래스 import
from dotenv import load_dotenv
import wandb
from functools import lru_cache
from torchvision.transforms import Resize, Compose, ToTensor
from functools import partial  # collate_fn에 인자 전달을 위한 패키지
from torch.utils.data._utils.pin_memory import pin_memory
from transformers import AutoTokenizer, AutoModelForCausalLM
from llava.utils import disable_torch_init
from accelerate import init_empty_weights
from accelerate import Accelerator 
from transformers import BitsAndBytesConfig
import cv2  # OpenCV를 활용한 빠른 이미지 저장
from huggingface_hub import notebook_login
load_dotenv()
os.environ["WANDB_API_KEY"] = ""
os.environ["HUGGING_FACE_HUB_TOKEN"] = ""
notebook_login()
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['CUDA_VISIBLE_DEVICES'] = '0,1,2,3' 
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if torch.cuda.device_count() > 1:
    print(f"🖥 Using {torch.cuda.device_count()} GPUs: [0, 1]")

    
cache_dir = "/home/cbn-gpu12/FNF/VLM/LLavA/dataset/cache"
os.makedirs(cache_dir, exist_ok=True)

🖥 Using 4 GPUs: [0, 1]


In [2]:
class CustomImageDataset(Dataset):
    def __init__(self, root_dir, full_image_dir, fold=None, split='train'):
        #self.trasform = transform
        self.root_dir = root_dir
        self.full_image_dir = full_image_dir
        self.fold = fold
        self.split = split

        if fold is None:
            self.images_dir = os.path.join(root_dir,'test','images')
            self.metadata_dir = os.path.join(root_dir, f'fold_{fold}', split, 'metadata')  # metadata 폴더 추가
        else:
            self.images_dir = os.path.join(root_dir, f'fold_{fold}', split,'images')
            self.metadata_dir = os.path.join(root_dir, f'fold_{fold}', split, 'metadata')  # metadata 폴더 추가
        print(f"\n Loading dataset from: {self.images_dir}")
        print(f" Using metadata from: {self.metadata_dir}")

        self.images = []
        self.metadata_cache = {}

        json_files = glob.glob(os.path.join(self.metadata_dir, "*.json"))

        with ThreadPoolExecutor(max_workers=8) as ecexutor:
            results = ecexutor.map(self._load_json, json_files)
        
        for filename, metadata in results:
            if metadata:
                self.metadata_cache[filename] = metadata
        
        view_folders = ['Lateral', 'Left', 'Right']  # 'Righ'로 수정
        self.images = []
        for view in view_folders:
            view_path = os.path.join(self.images_dir, view)
            if os.path.exists(view_path):
                print(f"Processing {view} folder...")
                png_files = glob.glob(os.path.join(view_path, "*.png"))
                for img_path in png_files:
                    img_name = os.path.basename(img_path)
                    filename = img_name.replace('.png', '')
                    if filename in self.metadata_cache:
                        metadata = self.metadata_cache[filename]
                        label = metadata.get('label', None)
                        self.images.append(
                            {'path': img_path, 'filename': img_name, 'label': label if label is not None else None}
                        )

        print(f'\nDataset loaded ({split}): Total samples = {len(self.images)}')

        if fold is not None:
            label_counts = {label:0 for label in range(1, 5)}
            for img in self.images:
                if img['label'] is not None:
                    label_counts[img['label']] += 1
            for label, count in sorted(label_counts.items()):
                print(f' - Garden Type {label}: {count}')

        if len(self.images) == 0:
            raise ValueError(f'No valid samples found in {self.images_dir}')
        
    
    def _load_json(self, json_path):
        try:
            with open(json_path, 'r') as f:
                metadata = json.load(f)
            filename = os.path.basename(json_path).replace('.json', '')
            return filename, metadata
        except Exception as e:
            print(f'Error loading JSON {json_path}: {e}')
            return None, None
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img_info = self.images[idx]

        try: 
            crop_image = Image.open(img_info['path']).convert('RGB')
        except Exception as e:
            print(f" Error loading image {img_info['path']}: {e}")
            crop_image = Image.new('RGB', (224,224))

        filename = img_info['filename']
        serial = filename.split('_')[0]

        if 'AP' in filename.upper():
            suffix = 'a000.dcm'
        elif 'LAT' in filename.upper():
            suffix = 't000.dcm'
        else:
            suffix = 'a000.dcm'

        full_img_path = os.path.join(self.full_image_dir, filename.split('_')[0], f"{filename.split('_')[0]}t000.dcm")

        try:
            dcm = pydicom.dcmread(full_img_path)
            full_image_array = dcm.pixel_array.astype(np.float32)  # floate32를 float32로 수정

            min_val,max_val = np.min(full_image_array), np.max(full_image_array)
            full_image_array = (full_image_array - min_val) / (max_val - min_val) if max_val != min_val else np.zeros_like(full_image_array)
            full_image_array = (full_image_array * 255).astype(np.uint8)
            full_image = Image.fromarray(full_image_array).convert('RGB')
        except Exception as e:
            print(f'Error loading DICOM file {full_img_path}:{e}')
            full_image = Image.new('RGB',(224,224))

        label = img_info.get('label')
        label_tensor = torch.tensor(label-1, dtype=torch.long) if label is not None else None

        return {
            'crop_image': crop_image,
            'full_image': full_image,
            'filename': filename,
            'label': label_tensor,
            'path': img_info['path']
        }

In [3]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

dataset_root = '/mnt/nas_backup/고효진/FNF/dataset'
    # full_image_dir는 이제 DICOM 데이터 폴더로 지정합니다.
full_image_dir = '/mnt/nas_backup/고효진/FNF/FNF_origin/FNF/Final_data/datasets/raw_internal'
result_dir = '/home/cbn-gpu12/FNF/VLM/LLaVA/datase'


model_name = "microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"
num_classes = 0

batch_size = 8
num_workers = 2
num_folds = 5 
start_time = time.time()
for fold in range(1, num_folds+1):
    train_dataset = CustomImageDataset(root_dir=dataset_root, full_image_dir=full_image_dir,fold=fold, split='train')
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)

    val_dataset = CustomImageDataset(root_dir=dataset_root, full_image_dir=full_image_dir,
                                           fold=fold, split='val')
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
endtime = time.time() - start_time

print(f'time for data loading : {endtime}')





 Loading dataset from: /mnt/nas_backup/고효진/FNF/dataset/fold_1/train/images
 Using metadata from: /mnt/nas_backup/고효진/FNF/dataset/fold_1/train/metadata
Processing Lateral folder...
Processing Left folder...
Processing Right folder...

Dataset loaded (train): Total samples = 8128
 - Garden Type 1: 1968
 - Garden Type 2: 348
 - Garden Type 3: 2488
 - Garden Type 4: 3324

 Loading dataset from: /mnt/nas_backup/고효진/FNF/dataset/fold_1/val/images
 Using metadata from: /mnt/nas_backup/고효진/FNF/dataset/fold_1/val/metadata
Processing Lateral folder...
Processing Left folder...
Processing Right folder...

Dataset loaded (val): Total samples = 2036
 - Garden Type 1: 484
 - Garden Type 2: 64
 - Garden Type 3: 608
 - Garden Type 4: 880

 Loading dataset from: /mnt/nas_backup/고효진/FNF/dataset/fold_2/train/images
 Using metadata from: /mnt/nas_backup/고효진/FNF/dataset/fold_2/train/metadata
Processing Lateral folder...
Processing Left folder...
Processing Right folder...

Dataset loaded (train): Total sam

In [4]:
# 데이터셋 정보 출력을 위한 코드
for fold in range(1, 6):
    print(f"\n{'='*50}")
    print(f"Fold {fold} 데이터셋 분석")
    print(f"{'='*50}")
    
    print(f"\n📊 전체 샘플 수: {len(train_dataset)}")
    
    # 클래스 분포 확인
    label_counts = {i: 0 for i in range(1, 5)}
    for item in train_dataset.images:
        if item['label'] is not None:
            label_counts[item['label']] += 1

    print('\n📈 클래스 분포:')
    for label, count in sorted(label_counts.items()):
        percentage = (count/len(train_dataset)*100) if len(train_dataset) > 0 else 0
        print(f"  - Garden Type {label}: {count}개 ({percentage:.1f}%)")
    
    # 이미지 정보 확인
    if len(train_dataset) > 0:
        sample = train_dataset[0]
        crop_img = sample['crop_image']
        full_img = sample['full_image']
        
        print('\n🖼️ 이미지 정보:')
        print(f'  - Crop 이미지:')
        print(f'    • 타입: {type(crop_img)}')
        print(f'    • 크기: {crop_img.size}')  # PIL Image는 size 사용
        print(f'  - Full 이미지:')
        print(f'    • 타입: {type(full_img)}')
        print(f'    • 크기: {full_img.size}')  # PIL Image는 size 사용

    # View별 분포 확인
    view_counts = {'Lateral': 0, 'Left': 0, 'Right': 0}
    for item in train_dataset.images:
        path = item['path']
        for view in view_counts.keys():
            if f'/{view}/' in path:
                view_counts[view] += 1
                break
                
    print('\n👁️ View별 이미지 수:')
    for view, count in sorted(view_counts.items()):
        percentage = (count/len(train_dataset)*100) if len(train_dataset) > 0 else 0
        print(f"  - {view}: {count}개 ({percentage:.1f}%)")
    
    print(f"\n📁 데이터셋 경로:")
    print(f"  - 이미지 경로: {train_dataset.images_dir}")
    print(f"  - 메타데이터 경로: {train_dataset.metadata_dir}")
    print(f"  - 전체 이미지 경로: {train_dataset.full_image_dir}")


Fold 1 데이터셋 분석

📊 전체 샘플 수: 8132

📈 클래스 분포:
  - Garden Type 1: 1976개 (24.3%)
  - Garden Type 2: 336개 (4.1%)
  - Garden Type 3: 2504개 (30.8%)
  - Garden Type 4: 3316개 (40.8%)

🖼️ 이미지 정보:
  - Crop 이미지:
    • 타입: <class 'PIL.Image.Image'>
    • 크기: (224, 224)
  - Full 이미지:
    • 타입: <class 'PIL.Image.Image'>
    • 크기: (1760, 2140)

👁️ View별 이미지 수:
  - Lateral: 4116개 (50.6%)
  - Left: 2064개 (25.4%)
  - Right: 1952개 (24.0%)

📁 데이터셋 경로:
  - 이미지 경로: /mnt/nas_backup/고효진/FNF/dataset/fold_5/train/images
  - 메타데이터 경로: /mnt/nas_backup/고효진/FNF/dataset/fold_5/train/metadata
  - 전체 이미지 경로: /mnt/nas_backup/고효진/FNF/FNF_origin/FNF/Final_data/datasets/raw_internal

Fold 2 데이터셋 분석

📊 전체 샘플 수: 8132

📈 클래스 분포:
  - Garden Type 1: 1976개 (24.3%)
  - Garden Type 2: 336개 (4.1%)
  - Garden Type 3: 2504개 (30.8%)
  - Garden Type 4: 3316개 (40.8%)

🖼️ 이미지 정보:
  - Crop 이미지:
    • 타입: <class 'PIL.Image.Image'>
    • 크기: (224, 224)
  - Full 이미지:
    • 타입: <class 'PIL.Image.Image'>
    • 크기: (1760, 2140)

👁️ View별 이미지 수:

In [5]:
'''class CustomVQADataset(Dataset):
    def __init__(self, image_dataset, save_dir=None):
       
        super(CustomVQADataset, self).__init__()
        self.image_dataset = image_dataset
        self.save_dir = save_dir
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),  # 모든 이미지를 224x224로 리사이즈
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        self.question = "What type of Garden fracture is shown in this X-ray image? Include explain for the answer"
        if save_dir:
            self.save_dataset()
    def _save_metadata(self, dataset_info):
        """메타데이터를 JSON 파일로 저장"""
        metadata_path = os.path.join(self.save_dir, 'metadata.json')
        try:
            metadata = {
                'info': {
                    'total_samples': len(dataset_info),
                    'date_updated': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
                    'question_template': self.question
                },
                'samples': dataset_info
            }
            
            with open(metadata_path, 'w', encoding='utf-8') as f:
                json.dump(metadata, f, indent=2, ensure_ascii=False)
                
        except Exception as e:
            print(f"메타데이터 저장 중 오류 발생: {e}")
            raise
    def save_dataset(self):
        """데이터셋을 지정된 디렉토리에 저장"""
        try:
            # 1. 디렉토리 생성 및 권한 확인
            os.makedirs(self.save_dir, exist_ok=True)
            images_dir = os.path.join(self.save_dir, "images")
            os.makedirs(images_dir, exist_ok=True)
            
            # 디렉토리 쓰기 권한 확인
            if not os.access(self.save_dir, os.W_OK):
                raise PermissionError(f"저장 디렉토리에 쓰기 권한이 없습니다: {self.save_dir}")
            
            print(f"\n저장 경로 확인:")
            print(f"- 메인 디렉토리: {self.save_dir}")
            print(f"- 이미지 디렉토리: {images_dir}")
            
            dataset_info = []
            print(f"\nVQA 데이터셋 저장 중... ({len(self.image_dataset)} 샘플)")
            
            # 2. 테스트 파일 저장
            test_file_path = os.path.join(self.save_dir, "test.txt")
            with open(test_file_path, 'w') as f:
                f.write("테스트 파일")
            if not os.path.exists(test_file_path):
                raise IOError("테스트 파일 저장에 실패했습니다.")
            
            # 3. 첫 번째 이미지 저장 테스트
            first_item = self.image_dataset[0]
            test_image_path = os.path.join(images_dir, "test_image.png")
            first_item['crop_image'].save(test_image_path)
            if not os.path.exists(test_image_path):
                raise IOError("테스트 이미지 저장에 실패했습니다.")
                
            print("✅ 초기 테스트 저장 성공")
            
            # 4. 본격적인 데이터셋 저장
            for idx in tqdm(range(len(self.image_dataset))):
                item = self.image_dataset[idx]
                
                # 이미지 저장
                sample_id = f"sample_{idx:06d}"
                crop_path = os.path.join(images_dir, f"{sample_id}_crop.png")
                full_path = os.path.join(images_dir, f"{sample_id}_full.png")
                
                # 이미지 저장 전 확인
                try:
                    item['crop_image'].save(crop_path)
                    item['full_image'].save(full_path)
                    
                    # 저장 확인
                    if not (os.path.exists(crop_path) and os.path.exists(full_path)):
                        raise IOError(f"이미지 저장 실패: {sample_id}")
                    
                except Exception as e:
                    print(f"\n⚠️ {sample_id} 이미지 저장 중 오류: {e}")
                    continue
                
                # 메타데이터 생성
                label = item['label'].item()
                garden_type = label + 1
                dataset_info.append({
                    'id': sample_id,
                    'crop_image': crop_path,
                    'full_image': full_path,
                    'question': self.question,
                    'answer': self._generate_detailed_answer(garden_type),
                    'label': label,
                    'garden_type': garden_type,
                    'original_filename': item.get('filename', '')
                })
                
                # 100개 단위로 중간 저장 및 확인
                if len(dataset_info) % 100 == 0:
                    self._save_metadata(dataset_info)
                    print(f"\n중간 저장 완료: {len(dataset_info)}개")
                    print(f"디렉토리 상태: {os.listdir(self.save_dir)}")
                    print(f"이미지 개수: {len(os.listdir(images_dir))}")
            
            # 최종 메타데이터 저장
            self._save_metadata(dataset_info)
            
            # 5. 최종 확인
            final_image_count = len(os.listdir(images_dir))
            print(f"\n✅ 저장 완료:")
            print(f"- 총 저장된 이미지: {final_image_count}")
            print(f"- 메타데이터 샘플 수: {len(dataset_info)}")
            print(f"- 저장 디렉토리 내용: {os.listdir(self.save_dir)}")
            
        except Exception as e:
            print(f"\n❌ 데이터셋 저장 중 오류 발생: {e}")
            raise

    def __len__(self):
        return len(self.image_dataset)
    
    def _generate_detailed_answer(self, garden_type):
        """Garden Type별 상세 설명을 생성하는 함수"""
        answers = {
            1: "This is a Garden Type I fracture. \
                The key features are:\
                - Incomplete fracture with valgus impaction,\
                - Minimal or no cortical disruption,\
                - Generally stable configuration.\
                The fracture line may be subtle, often appearing as trabecular impaction rather than a clear break.",
            
            2: "This is a Garden Type II fracture.\
                The characteristic features include:\
                - Complete fracture without displacement,\
                - Minimal disruption of trabecular pattern,\
                - No significant angulation or rotation.\
                Despite the complete fracture, the bone fragments remain properly aligned, making it a stable fracture.",
            
            3: "This is a Garden Type III fracture.\
                The diagnostic features include:\
                - Complete fracture with partial displacement,\
                - Some disruption of trabecular alignment,\
                - Cortical contact partially maintained but with angulation.\
                There may be early signs of femoral head malalignment, increasing the risk of instability and avascular necrosis.",
            
            4: "This is a Garden Type IV fracture.\
                The distinctive features include:\
                - Complete fracture with full displacement,\
                - No cortical contact between fragments,\
                - Severe disruption of trabecular and anatomical alignment.\
                The femoral head is completely separated from the shaft, significantly increasing the risk of avascular necrosis."
        }
        return answers.get(garden_type, "Unknown fracture type")

    def __getitem__(self, idx):
        item = self.image_dataset[idx]
        
        # 이미지 변환 전 크기 출력
        print(f"Before transform - Crop image size: {item['crop_image'].size}, Full image size: {item['full_image'].size}")
        
        # 이미지 변환
        crop_image = self.transform(item['crop_image'])
        full_image = self.transform(item['full_image'])
        
        # 변환 후 텐서 크기 출력
        print(f"After transform - Crop image shape: {crop_image.shape}, Full image shape: {full_image.shape}")
        
        # 라벨 처리
        label = item['label']
        garden_type = label.item() + 1
        answer = self._generate_detailed_answer(garden_type)

        return {
            'crop_image': crop_image,
            'full_image': full_image,
            'question': self.question,
            'answer': answer,
            'label': label
        }
    '''

class CustomVQADataset(Dataset):
    def __init__(self, image_dataset, save_dir=None):
        super(CustomVQADataset, self).__init__()
        self.image_dataset = image_dataset
        self.save_dir = save_dir
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        
        # 가이드라인을 포함한 프롬프트 형태의 질문
        self.question = """Analyze the type of femoral neck fracture (Garden classification) shown in this X-ray image and determine which Garden type it belongs to.\
            Consider the following criteria in your response:
            1: "This is a Garden Type I fracture. \
                    The key features are:\
                    - Incomplete fracture with valgus impaction,\
                    - Minimal or no cortical disruption,\
                    - Generally stable configuration.\
                    The fracture line may be subtle, often appearing as trabecular impaction rather than a clear break.",
                
                2: "This is a Garden Type II fracture.\
                    The characteristic features include:\
                    - Complete fracture without displacement,\
                    - Minimal disruption of trabecular pattern,\
                    - No significant angulation or rotation.\
                    Despite the complete fracture, the bone fragments remain properly aligned, making it a stable fracture.",
                
                3: "This is a Garden Type III fracture.\
                    The diagnostic features include:\
                    - Complete fracture with partial displacement,\
                    - Some disruption of trabecular alignment,\
                    - Cortical contact partially maintained but with angulation.\
                    There may be early signs of femoral head malalignment, increasing the risk of instability and avascular necrosis.",
                
                4: "This is a Garden Type IV fracture.\
                    The distinctive features include:\
                    - Complete fracture with full displacement,\
                    - No cortical contact between fragments,\
                    - Severe disruption of trabecular and anatomical alignment.\
                    The femoral head is completely separated from the shaft, significantly increasing the risk of avascular necrosis."""

        if save_dir:
            self.save_dataset()

    def save_dataset(self):
        """데이터셋 기본 정보 저장"""
        try:
            os.makedirs(self.save_dir, exist_ok=True)
            
            # 기본 정보 수집
            dataset_info = [{
                'id': idx,
                'image_path': item['path'],
                'label': int(item['label'])
            } for idx, item in enumerate(self.image_dataset.images)]
            
            base_info = {
                'total_samples': len(dataset_info),
                'date_created': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
                'question_template': self.question,
                'samples': dataset_info
            }
            
            # dataset_info.json으로 저장
            info_path = os.path.join(self.save_dir, 'custom_qa.json')
            with open(info_path, 'w', encoding='utf-8') as f:
                json.dump(base_info, f, ensure_ascii=False)
            
            print(f"✅ 데이터셋 기본 정보 저장 완료: 총 {len(dataset_info)}개 샘플")
            
        except Exception as e:
            print(f"❌ 데이터셋 정보 저장 중 오류 발생: {e}")
            raise
    def __len__(self):
        return len(self.image_dataset)
    def __getitem__(self, idx):
        item = self.image_dataset[idx]
        
        crop_image = self.transform(item['crop_image'])
        full_image = self.transform(item['full_image'])
        label = item['label']

        answer = f"This is a Garden Type {label+1} fracture."
    
        return {
            'crop_image': crop_image,
            'full_image': full_image,
            'question': self.question,
            'answer': answer,  # answer 추가
            'label': label,  # 평가용
            'id': f"sample_{idx:06d}"  # sample_id 추가
        }

In [6]:
class DataCollator:
    def __init__(self, tokenizer, split, conversation_template, pad_token_id, image_processor, model_config):
        self.tokenizer = tokenizer
        self.split = split
        self.conversation_template = conversation_template
        self.pad_token_id = pad_token_id
        self.image_processor = image_processor
        self.model_config = model_config

    def __call__(self, rows):
        if not isinstance(rows, list):
            rows = [rows]
        
        if self.split == "train":
            return self._collate_train(rows)
        elif self.split == "val":
            return self._collate_test(rows)

    def _collate_train(self, rows):
        train_input_ids_list = []
        train_labels_list = []
        train_images = []

        for row in rows:
            try:
                # 데이터 추출 및 크기 확인
                crop_image = row['crop_image']
                full_image = row['full_image']
                
                print(f"In DataCollator - Crop image shape: {crop_image.shape}, Full image shape: {full_image.shape}")
                
                # 이미지 처리 - 두 이미지를 수직으로 연결
                combined_image = torch.cat([crop_image, full_image], dim=1)  # height 방향으로 연결
                print(f"Combined image shape: {combined_image.shape}")
                
                # 이미지 프로세서를 통한 추가 처리
                processed_image = combined_image.permute(1, 2, 0).numpy()  # CHW -> HWC
                processed_image = Image.fromarray((processed_image * 255).astype(np.uint8))
                processed_image = self.image_processor.preprocess(
                    processed_image,
                    return_tensors="pt"
                )["pixel_values"][0]
                
                print(f"Final processed image shape: {processed_image.shape}")
                
                train_images.append(processed_image)

                # 나머지 코드는 동일...
                question = row['question']
                answer = row['answer']
                
                # 이미지 토큰을 하나로 변경
                question = question.replace(DEFAULT_IMAGE_TOKEN, '').strip()
                question = DEFAULT_IMAGE_TOKEN + '\n' + question

                # 대화 형식으로 변환
                conv = self.conversation_template.copy()
                conv.append_message(conv.roles[0], question)
                conv.append_message(conv.roles[1], None)
                prefix = conv.get_prompt()

                conv = self.conversation_template.copy()
                conv.append_message(conv.roles[0], question)
                conv.append_message(conv.roles[1], answer)
                full = conv.get_prompt()

                # 토큰화
                prefix = tokenizer_image_token(prefix, self.tokenizer, IMAGE_TOKEN_INDEX, return_tensors="pt")
                full = tokenizer_image_token(full, self.tokenizer, IMAGE_TOKEN_INDEX, return_tensors="pt")

                prefix_length = prefix.size(0)
                train_input_ids = full
                train_labels = full.clone()
                train_labels[:prefix_length] = -100

                train_input_ids_list.append(train_input_ids)
                train_labels_list.append(train_labels)
                
            except Exception as e:
                print(f"Error processing row: {e}")
                print(f"Row keys: {row.keys()}")
                if 'crop_image' in row and 'full_image' in row:
                    print(f"Crop image type: {type(row['crop_image'])}")
                    print(f"Full image type: {type(row['full_image'])}")
                continue

        if not train_input_ids_list:
            raise ValueError("No valid data found in the batch")

        # 패딩 처리
        pad_value = -114514
        train_input_ids = pad_sequence(train_input_ids_list, batch_first=True, padding_value=pad_value)
        train_labels = pad_sequence(train_labels_list, batch_first=True, padding_value=pad_value)
        train_attention_mask = (train_input_ids != pad_value).long()

        train_input_ids[train_input_ids == pad_value] = self.pad_token_id
        train_labels[train_labels == pad_value] = self.pad_token_id

        # 이미지 스택 및 타입 변환
        train_images = torch.stack(train_images).to(torch.bfloat16)
        sample_ids = [row['id'] for row in rows]
        return {
        "input_ids": train_input_ids,
        "labels": train_labels,
        "attention_mask": train_attention_mask,
        "images": train_images,
        #"sample_ids": sample_ids  
        "metadata": {"sample_ids": sample_ids}
    }
    def _collate_test(self, rows):
        pass

In [ ]:
'''class CustomTrainer(Trainer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.best_loss = float('inf')
        self.last_step_time = time.time()  # 시간 측정을 위한 초기화
    
    def training_step(self, model, inputs, num_items_in_batch=None):  # num_items_in_batch 파라미터 추가
        """학습 스텝 수행 및 시간 측정"""
        # 스텝별 시간 측정
        current_time = time.time()
        elapsed = current_time - self.last_step_time
        self.last_step_time = current_time
        
        # 남은 시간 추정
        steps_done = self.state.global_step
        total_steps = len(self.train_dataset) * self.args.num_train_epochs
        remaining_steps = total_steps - steps_done
        estimated_time = remaining_steps * elapsed
        
        print(f"\n스텝 {steps_done} 처리 시간: {elapsed:.2f}초")
        print(f"예상 남은 시간: {estimated_time/3600:.2f}시간")
        
        # GPU 메모리 상태 출력
        if torch.cuda.is_available():
            for i in range(torch.cuda.device_count()):
                print(f"GPU {i} 메모리: {torch.cuda.memory_allocated(i)/1024**2:.1f}MB / {torch.cuda.memory_reserved(i)/1024**2:.1f}MB")
        
        # 원래의 training_step 호출
        return super().training_step(model, inputs, num_items_in_batch)
    
    def save_model(self, output_dir=None, _internal_call=False):
        """체크포인트 저장 시 추가 정보도 함께 저장"""
        output_dir = output_dir if output_dir is not None else self.args.output_dir
        os.makedirs(output_dir, exist_ok=True)
        
        # 모델 상태 저장
        self.model.save_pretrained(output_dir)
        
        # 옵티마이저 상태 저장
        torch.save(self.optimizer.state_dict(), os.path.join(output_dir, "optimizer.pt"))
        
        # 학습 상태 저장
        training_state = {
            "epoch": self.state.epoch,
            "global_step": self.state.global_step,
            "best_loss": self.best_loss,
            "train_loss": self.state.log_history[-1]["loss"] if self.state.log_history else None,
        }
        torch.save(training_state, os.path.join(output_dir, "training_state.pt"))
        
        # 스케줄러 상태 저장
        if self.lr_scheduler is not None:
            torch.save(self.lr_scheduler.state_dict(), os.path.join(output_dir, "scheduler.pt"))
            
    def load_checkpoint(self, checkpoint_dir):
        """체크포인트에서 학습 재개"""
        try:
            # 모델 로드
            self.model = self.model.from_pretrained(checkpoint_dir)
            
            # 옵티마이저 상태 로드
            optimizer_path = os.path.join(checkpoint_dir, "optimizer.pt")
            if os.path.exists(optimizer_path):
                self.optimizer.load_state_dict(torch.load(optimizer_path))
                
            # 학습 상태 로드
            training_state_path = os.path.join(checkpoint_dir, "training_state.pt")
            if os.path.exists(training_state_path):
                training_state = torch.load(training_state_path)
                self.state.epoch = training_state["epoch"]
                self.state.global_step = training_state["global_step"]
                self.best_loss = training_state["best_loss"]
                
            # 스케줄러 상태 로드
            scheduler_path = os.path.join(checkpoint_dir, "scheduler.pt")
            if os.path.exists(scheduler_path) and self.lr_scheduler is not None:
                self.lr_scheduler.load_state_dict(torch.load(scheduler_path))
                
            print(f"✅ 체크포인트 로드 완료: {checkpoint_dir}")
            
        except Exception as e:
            print(f"⚠️ 체크포인트 로드 중 오류 발생: {e}")
            raise'''

class CustomTrainer(Trainer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.accelerator = Accelerator(
            gradient_accumulation_steps=self.args.gradient_accumulation_steps,
            mixed_precision='bf16'  # 기존 설정과 일치
        )
        self.best_loss = float('inf')
        self.last_step_time = time.time()
        self.last_checkpoint_step = 0
        self.memory_threshold = 0.85  # GPU 메모리 사용률 임계값
        self.checkpoint_frequency = 50
        self.vqa_save_dir = "/mnt/nas_backup/고효진/FNF/dataset/VQA_dataset"
        self.custom_qa_path = os.path.join(self.vqa_save_dir, 'custom_qa.json')
        if not wandb.run:
            wandb.init(
                project="llava-med-training",
                name=f"fnf-classification-{datetime.now().strftime('%Y%m%d-%H%M')}",
                config={
                    "model_name": "llava-med-v1.5-mistral-7b",
                    "batch_size": self.args.per_device_train_batch_size,
                    "learning_rate": self.args.learning_rate,
                    "epochs": self.args.num_train_epochs,
                }
            )
    def _check_memory_usage(self):
        """GPU 메모리 사용량 체크"""
        if torch.cuda.is_available():
            for i in range(torch.cuda.device_count()):
                memory_allocated = torch.cuda.memory_allocated(i)
                memory_reserved = torch.cuda.memory_reserved(i)
                memory_total = torch.cuda.get_device_properties(i).total_memory
                memory_ratio = memory_allocated / memory_total
                
                if memory_ratio > self.memory_threshold:
                    return True
        return False
    def _clear_cuda_memory(self):
        """CUDA 메모리를 더 철저하게 정리"""
        if torch.cuda.is_available():
            # 모델을 CPU로 이동
            self.model.cpu()
            if hasattr(self, 'optimizer'):
                self.optimizer.state = defaultdict(dict)  # 옵티마이저 상태 초기화
            
            # 각 GPU별로 캐시 정리
            for i in range(torch.cuda.device_count()):
                with torch.cuda.device(i):
                    torch.cuda.empty_cache()
                    torch.cuda.reset_peak_memory_stats()
            
            # 가비지 컬렉션 강제 실행
            gc.collect()
            
            print("✅ CUDA 메모리 완전 정리 완료")
    def save_dataset(self):
        """초기 메타데이터 빠르게 저장"""
        try:
            os.makedirs(self.save_dir, exist_ok=True)
            
            # 기본 메타데이터만 빠르게 저장
            dataset_info = [{
                'id': f"sample_{idx:06d}",
                'image_path': item['path'],
                'label': int(item['label'])
            } for idx, item in enumerate(self.image_dataset)]
            
            metadata = {
                'info': {
                    'total_samples': len(dataset_info),
                    'date_updated': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
                    'question_template': self.question
                },
                'samples': dataset_info
            }
            
            with open(os.path.join(self.save_dir, 'metadata.json'), 'w', encoding='utf-8') as f:
                json.dump(metadata, f, ensure_ascii=False)
            
            print(f"✅ 메타데이터 저장 완료: 총 {len(dataset_info)}개 샘플")
            
        except Exception as e:
            print(f"❌ 메타데이터 저장 중 오류 발생: {e}")
            raise

    def _save_emergency_checkpoint(self):
        """긴급 체크포인트 저장"""
        checkpoint_dir = os.path.join(self.args.output_dir, f"emergency_checkpoint_{self.state.global_step}")
        os.makedirs(checkpoint_dir, exist_ok=True)
        
        try:
            print(f"긴급 체크포인트 저장 시도 중: {checkpoint_dir}")
            
            # 1. PEFT 모델 저장
            self.model.save_pretrained(checkpoint_dir)
            print("✓ PEFT 어댑터 저장 완료")
            
            # 2. 옵티마이저 상태 저장
            optimizer_state = {
                key: value.cpu() if torch.is_tensor(value) else value
                for key, value in self.optimizer.state_dict().items()
            }
            torch.save(optimizer_state, os.path.join(checkpoint_dir, "optimizer.pt"))
            print("✓ 옵티마이저 상태 저장 완료")
            
            # 3. 학습 상태 저장
            training_state = {
                "epoch": self.state.epoch,
                "global_step": self.state.global_step,
                "best_loss": self.best_loss,
                "train_loss": self.state.log_history[-1]["loss"] if self.state.log_history else None,
            }
            torch.save(training_state, os.path.join(checkpoint_dir, "training_state.pt"))
            print("✓ 학습 상태 저장 완료")
            
            print(f"✅ 긴급 체크포인트 저장 완료: {checkpoint_dir}")
            return checkpoint_dir
            
        except Exception as e:
            print(f"⚠️ 긴급 체크포인트 저장 중 오류 발생: {e}")
            raise
    
    def _resume_from_checkpoint(self, checkpoint_dir):
        """체크포인트에서 학습 재개"""
        try:
            print(f"체크포인트 {checkpoint_dir}에서 복구 시도 중...")
            
            # 1. 메모리 정리
            self._clear_cuda_memory()
            self._print_gpu_status()
            
            # 2. PEFT 모델 로드
            print("PEFT 어댑터 로드 중...")
            self.model = PeftModel.from_pretrained(
                self.model.get_base_model(),
                checkpoint_dir,
                is_trainable=True,
                device_map="auto"
            )
            print("✓ PEFT 어댑터 로드 완료")
            
            # 3. 옵티마이저 상태 로드
            optimizer_path = os.path.join(checkpoint_dir, "optimizer.pt")
            if os.path.exists(optimizer_path):
                print("옵티마이저 상태 로드 중...")
                optimizer_state = torch.load(optimizer_path, map_location="cpu")
                self.optimizer.load_state_dict(optimizer_state)
                print("✓ 옵티마이저 상태 로드 완료")
            
            # 4. 학습 상태 로드
            training_state_path = os.path.join(checkpoint_dir, "training_state.pt")
            if os.path.exists(training_state_path):
                print("학습 상태 로드 중...")
                training_state = torch.load(training_state_path, map_location="cpu")
                self.state.epoch = training_state["epoch"]
                self.state.global_step = training_state["global_step"]
                self.best_loss = training_state["best_loss"]
                print("✓ 학습 상태 로드 완료")
            
            print(f"✅ 체크포인트 {checkpoint_dir}에서 학습 재개 성공")
            return True
            
        except Exception as e:
            print(f"⚠️ 체크포인트 로드 실패: {e}")
            print("💡 체크포인트 복구 실패 시 조치:")
            print("1. 모든 GPU 메모리 정리")
            print("2. 프로세스 재시작 필요할 수 있음")
            print(f"3. 체크포인트 디렉토리 확인: {checkpoint_dir}")
            return False

    def training_step(self, model, inputs, num_items_in_batch=None):
        try:
            # 현재 스텝이 checkpoint_frequency의 배수이거나 메모리 사용량이 높을 때 체크포인트 저장
            current_step = self.state.global_step
            if (current_step % self.checkpoint_frequency == 0 and current_step > 0) or \
               self._check_memory_usage():
                checkpoint_dir = self._save_emergency_checkpoint()
                self._clear_cuda_memory()
                if not self._resume_from_checkpoint(checkpoint_dir):
                    raise RuntimeError("체크포인트 저장 후 복구 실패")
            with self.accelerator.accumulate(model):
                metadata = inputs.pop("metadata", {})
                sample_ids = metadata.get("sample_ids", [])
                inputs = {k: v.to(self.accelerator.device) if torch.is_tensor(v) else v 
                         for k, v in inputs.items()}
                outputs = super().training_step(model, inputs, num_items_in_batch)
                if self.state.global_step % 10 == 0:  # 10 스텝마다
                    try:
                        model.eval()
                        with torch.no_grad():
                            generation_outputs = model.generate(
                                input_ids=inputs['input_ids'],
                                attention_mask=inputs['attention_mask'],
                                images=inputs['images'],
                                max_new_tokens=512,
                                do_sample=True,
                                temperature=0.7
                            )
                        model.train()
                        generated_texts = self.tokenizer.batch_decode(generation_outputs, skip_special_tokens=True)
                        
                        if sample_ids:
                            self._update_vqa_dataset(
                                sample_ids=sample_ids,
                                predictions=generated_texts
                            )
                    except Exception as gen_error:
                        print(f"텍스트 생성 중 오류 발생: {gen_error}")
                        # 생성 오류가 발생해도 학습은 계속 진행

                if hasattr(outputs, 'generated_text'):
                    # wandb에 로깅
                    wandb.log({
                        "predictions": [
                            wandb.Image(img, caption=f"Prediction: {pred}")
                            for img, pred in zip(inputs['images'], generated_texts)
                        ],
                        "train/gpu_memory_allocated": torch.cuda.memory_allocated()/1024**2,
                        "train/gpu_memory_reserved": torch.cuda.memory_reserved()/1024**2,
                        "train/step_time": time.time() - self.last_step_time,
                    }, step=self.state.global_step)
                    
                

                # 현재 GPU 메모리 상태 출력
                if self.state.global_step % 10 == 0:  # 기존 GPU 상태 출력 주기와 동일하게 설정
                    wandb.log({
                        "train/gpu_memory_allocated": torch.cuda.memory_allocated()/1024**2,
                        "train/gpu_memory_reserved": torch.cuda.memory_reserved()/1024**2,
                        "train/step_time": time.time() - self.last_step_time,
                    }, step=self.state.global_step)
                    
                return outputs
            
        except RuntimeError as e:
            if "out of memory" in str(e):
                # 기존 OOM 처리 로직 유지
                checkpoint_dir = self._save_emergency_checkpoint()
                self._clear_cuda_memory()
                self._print_gpu_status()
                
                # wandb에 OOM 이벤트 로깅
                wandb.alert(
                    title="OOM Error",
                    text=f"Out of memory at step {self.state.global_step}. Emergency checkpoint saved."
                )
                
                if self._resume_from_checkpoint(checkpoint_dir):
                    return super().training_step(model, inputs, num_items_in_batch)
                else:
                    raise RuntimeError("체크포인트에서 복구 실패")
            else:
                raise e
    def _update_vqa_dataset(self, sample_ids, predictions):
        """VQA 데이터셋 메타데이터에 모델 답변 저장"""
        try:
            # 메타데이터 파일 읽기
            with open(self.custom_qa_path, 'r', encoding='utf-8') as f:
                custom_qa = json.load(f)
            
            # 각 샘플에 대한 예측 결과 업데이트
            for sample_id, prediction in zip(sample_ids, predictions):
                for sample in custom_qa['samples']:
                    if sample['id'] == sample_id:
                        sample['model_answer'] = prediction
                        sample['prediction_step'] = self.state.global_step
                        sample['prediction_loss'] = float(self.state.log_history[-1]["loss"]) if self.state.log_history else None
                        break
            
            # 업데이트된 메타데이터 저장
            with open(self.custom_qa_path, 'w', encoding='utf-8') as f:
                json.dump(custom_qa, f, indent=2, ensure_ascii=False)
                
        except Exception as e:
            print(f"⚠️ VQA 메타데이터 업데이트 중 오류 발생: {e}")
            
    def _print_gpu_status(self):
        """GPU 상태 출력"""
        if torch.cuda.is_available():
            for i in range(torch.cuda.device_count()):
                allocated = torch.cuda.memory_allocated(i)/1024**2
                reserved = torch.cuda.memory_reserved(i)/1024**2
                print(f"\nGPU {i}:")
                print(f"- 할당된 메모리: {allocated:.1f}MB")
                print(f"- 예약된 메모리: {reserved:.1f}MB")



In [9]:
def save_final_model(trainer, save_dir, model_name):
    """최종 학습된 모델 저장"""
    try:
        base_path = os.path.join(save_dir, model_name)
        os.makedirs(base_path, exist_ok=True)
        
        print("\n Start saving final model...")
        
        # 1. PEFT 모델 저장
        print("1. Saving PEFT model...")
        trainer.model.save_pretrained(base_path)
        
        # 2. 토크나이저 저장
        print("2. Saving tokenizer...")
        trainer.tokenizer.save_pretrained(base_path)
        
        # 3. 모델 병합 (PEFT -> 완전한 모델)
        print("3. Merging PEFT model...")
        base_model = AutoModelForCausalLM.from_pretrained(
            'microsoft/llava-med-v1.5-mistral-7b',
            torch_dtype=torch.float16,
            low_cpu_mem_usage=True
        )
        merged_model = PeftModel.from_pretrained(base_model, base_path).merge_and_unload()
        
        # 4. 병합된 모델 저장
        print("4. Saving final merged model...")
        merged_path = os.path.join(save_dir, f"{model_name}_merged")
        merged_model.save_pretrained(merged_path)
        trainer.tokenizer.save_pretrained(merged_path)
        
        # 5. HuggingFace Hub에 업로드 (선택적)
        print("5. Uploading model on HuggingFace Hub...")
        merged_model.push_to_hub(f"sarahyo941/{model_name}")
        trainer.tokenizer.push_to_hub(f"sarahyo941/{model_name}")
        
        # 6. wandb에 최종 모델 정보 저장
        wandb.run.summary.update({
            "final_model_path": base_path,
            "merged_model_path": merged_path,
            "huggingface_model": f"sarahyo941/{model_name}",
        })
        
        # 7. 설정 파일 저장
        config = {
            "model_name": model_name,
            "base_model": "microsoft/llava-med-v1.5-mistral-7b",
            "training_args": trainer.args.to_dict(),
            "creation_date": datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            "final_loss": trainer.state.best_metric,
            "total_steps": trainer.state.global_step,
            "total_epochs": trainer.state.num_train_epochs,
        }
        
        with open(os.path.join(base_path, "training_config.json"), "w") as f:
            json.dump(config, f, indent=2)
            
        print("\n✅ Model saved!")
        print(f"- PEFT model: {base_path}")
        print(f"- Merged model: {merged_path}")
        print(f"- HuggingFace Hub: sarahyo941/{model_name}")
        
    except Exception as e:
        print(f"\n⚠️ Error occured when saving model : {e}")
        raise
    finally:
        wandb.finish()



In [10]:


# GPU 메모리 설정
torch.cuda.empty_cache()
torch.cuda.set_per_process_memory_fraction(0.95)  # GPU 메모리의 95% 사용

# 환경 변수 설정
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['CUDA_VISIBLE_DEVICES'] = '0,1,2,3'  # 사용할 GPU 지정


quantization_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
    llm_int8_has_fp16_weight=False
)

# 모델 설정
model_path = "microsoft/llava-med-v1.5-mistral-7b"
model_base = "llava-med-v1.5-mistral-7b"
model_name = "llava-med-v1.5-mistral-7b"

# 토크나이저 초기화
tokenizer = LlamaTokenizer.from_pretrained(
    model_path,
    use_fast=False,
    trust_remote_code=True,
    padding_side="right",
    model_max_length=4096
)


# 모델 초기화 - 메모리 효율적인 방식
model = LlavaMistralForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    use_flash_attention_2=False,  # Flash Attention 2 비활성화
    device_map="auto",  # 자동으로 여러 GPU에 분산
    trust_remote_code=True,
    quantization_config=quantization_config,  # 8bit 양자화 적용
    max_memory={  # GPU 메모리 제한 증가
        0: "20GB",
        1: "20GB", 
        2: "20GB",
        3: "20GB"
    }
)


# Vision tower 초기화
vision_tower = model.get_vision_tower()
if not vision_tower.is_loaded:
    vision_tower.load_model()
vision_tower.to(dtype=torch.float16)

# 이미지 프로세서 초기화
image_processor = vision_tower.image_processor
context_len = model.config.max_position_embeddings

# 대화 템플릿 설정
conv = conv_templates["mistral_instruct"]

# DataCollator 설정
collate_fn = DataCollator(
    tokenizer=tokenizer,
    split="train",
    conversation_template=conv,
    pad_token_id=tokenizer.pad_token_id,
    image_processor=image_processor,
    model_config=model.config
)
# 메모리 사용량 확인
def print_gpu_memory():
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.memory_allocated(i)/1024**2:.2f}MB allocated, {torch.cuda.memory_reserved(i)/1024**2:.2f}MB reserved")

print_gpu_memory()

The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some weights of the model checkpoint at microsoft/llava-med-v1.5-mistral-7b were not used when initializing LlavaMistralForCausalLM: {'model.vision_tower.vision_tower.vision_model.encoder.layers.16.mlp.fc2.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.20.self_attn.out_proj.bias', 'model.vision_tower.vision_tower.vision_model.encoder.layers.15.self_attn.q_proj.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.3.layer_norm1.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.20.layer_norm1.bias', 'model.vision_tower.vision_tower.vision_model.encoder.layers.1.self_attn.k_proj.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.1.layer_norm2.bias', 'model.vision_tower.vision_tower.vision_model.encoder.layers.1.self_attn.out_proj.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.20.self_attn.k_proj.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.7.self_attn.v_proj.bias',

GPU 0: 1499.08MB allocated, 1582.00MB reserved
GPU 1: 1665.44MB allocated, 1702.00MB reserved
GPU 2: 1665.44MB allocated, 1722.00MB reserved
GPU 3: 2351.85MB allocated, 2382.00MB reserved


In [11]:
torch.cuda.empty_cache()
gc.collect()
model.train()

model.gradient_checkpointing_enable()

model = prepare_model_for_kbit_training(model)


lora_config = LoraConfig(
    r=8,
    target_modules=["k_proj", "q_proj", "v_proj", "out_proj"],
    lora_alpha=16
)

peft_model = get_peft_model(model, lora_config, "default")
peft_model.print_trainable_parameters()

trainable params: 6,291,456 || all params: 7,572,510,720 || trainable%: 0.0831


In [ ]:
save_dir = "/mnt/nas_backup/고효진/FNF/dataset/VQA_dataset"
vqa_rad_dataset_train = CustomVQADataset(
    image_dataset=train_dataset,
    save_dir=save_dir  # 저장이 필요한 경우에만 지정
)

training_args = TrainingArguments(
    output_dir="trained_llava-med",
    report_to="wandb",
    run_name=f"fnf-classification-{datetime.now().strftime('%Y%m%d-%H%M')}",  # wandb run name 추가
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    logging_steps=10,
    learning_rate=2e-5,
    logging_strategy="steps",
    save_strategy="steps",
    save_steps=200,
    evaluation_strategy="steps",
    eval_steps=200,
    save_total_limit=3,
    num_train_epochs=5,
    warmup_ratio=0.03,
    weight_decay=0.01,
    remove_unused_columns=False,
    gradient_checkpointing=True,
    fp16=False,
    bf16=True,
    optim='paged_adamw_8bit',
    label_names=["labels"],
    dataloader_num_workers=8,
    dataloader_pin_memory=True,
    dataloader_prefetch_factor=4)

'''trainer = CustomTrainer(
    model=peft_model,
    tokenizer=tokenizer,  # tokenizer 추가
    args=training_args,
    train_dataset=vqa_rad_dataset_train,
    eval_dataset=vqa_rad_dataset_train,
    data_collator=collate_fn
)'''

trainer = CustomTrainer(
    model=peft_model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=vqa_rad_dataset_train,
    eval_dataset=vqa_rad_dataset_train,
    data_collator=collate_fn
)
model, optimizer, train_dataloader = trainer.accelerator.prepare(
    trainer.model, 
    trainer.optimizer,
    trainer.get_train_dataloader()
)
# wandb 초기화 및 설정
wandb.init(
    project="llava-med-training",
    config={
        "model_name": "llava-med-v1.5-mistral-7b",
        "architecture": "mistral-7b",
        "dataset": "vqa_rad"
    }
)

latest_checkpoint = None
latest_checkpoint = None
if os.path.exists(training_args.output_dir):
    checkpoints = [d for d in os.listdir(training_args.output_dir) 
                  if d.startswith("checkpoint-")]
    if checkpoints:
        latest_checkpoint = max(checkpoints, key=lambda x: int(x.split("-")[1]))
        latest_checkpoint = os.path.join(training_args.output_dir, latest_checkpoint)
        print(f"최근 체크포인트를 발견했습니다: {latest_checkpoint}")
        trainer.load_checkpoint(latest_checkpoint)

# 학습 실행 코드 수정
try:
    trainer.train(resume_from_checkpoint=latest_checkpoint)
except Exception as e:
    print(f"학습 중 오류 발생: {e}")
    trainer.save_model(os.path.join(training_args.output_dir, "error_checkpoint"))
finally:
    # 최종 모델 저장
    save_final_model(
        trainer=trainer,
        save_dir="/mnt/nas_backup/고효진/FNF/dataset/VQA_dataset",
        model_name="llava-med-v1.5-mistral-7b-fnf"
    )


    

✅ 데이터셋 기본 정보 저장 완료: 총 8132개 샘플


wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: sarahyo941 (sarahyo941-university-of-ulsan) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In DataCollator - Crop image shape: torch.Size([3, 224, 224]), Full image shape: torch.Size([3, 224, 224])
Combined image shape: torch.Size([3, 448, 224])
Final processed image shape: torch.Size([3, 336, 336])
In DataCollator - Crop image shape: torch.Size([3, 224, 224]), Full image shape: torch.Size([3, 224, 224])
Combined image shape: torch.Size([3, 448, 224])
Final processed image shape: torch.Size([3, 336, 336])
In DataCollator - Crop image shape: torch.Size([3, 224, 224]), Full image shape: torch.Size([3, 224, 224])
Combined image shape: torch.Size([3, 448, 224])
Final processed image shape: torch.Size([3, 336, 336])
In DataCollator - Crop image shape: torch.Size([3, 224, 224]), Full image shape: torch.Size([3, 224, 224])
Combined image shape: torch.Size([3, 448, 224])
Final processed image shape: torch.Size([3, 336, 336])
In DataCollator - Crop image shape: torch.Size([3, 224, 224]), Full image shape: torch.Size([3, 224, 224])
Combined image shape: torch.Size([3, 448, 224])
Final

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


긴급 체크포인트 저장 시도 중: trained_llava-med/emergency_checkpoint_0
✓ PEFT 어댑터 저장 완료
✓ 옵티마이저 상태 저장 완료
✓ 학습 상태 저장 완료
✅ 긴급 체크포인트 저장 완료: trained_llava-med/emergency_checkpoint_0
✅ CUDA 메모리 완전 정리 완료

GPU 0:
- 할당된 메모리: 2219.9MB
- 예약된 메모리: 2370.0MB

GPU 1:
- 할당된 메모리: 3068.1MB
- 예약된 메모리: 3326.0MB

GPU 2:
- 할당된 메모리: 5563.8MB
- 예약된 메모리: 5868.0MB

GPU 3:
- 할당된 메모리: 7362.1MB
- 예약된 메모리: 7652.0MB
체크포인트 trained_llava-med/emergency_checkpoint_0에서 복구 시도 중...
✅ CUDA 메모리 완전 정리 완료

GPU 0:
- 할당된 메모리: 2219.9MB
- 예약된 메모리: 2370.0MB

GPU 1:
- 할당된 메모리: 3068.1MB
- 예약된 메모리: 3326.0MB

GPU 2:
- 할당된 메모리: 5563.8MB
- 예약된 메모리: 5868.0MB

GPU 3:
- 할당된 메모리: 7362.1MB
- 예약된 메모리: 7652.0MB
PEFT 어댑터 로드 중...
✓ PEFT 어댑터 로드 완료
옵티마이저 상태 로드 중...
✓ 옵티마이저 상태 로드 완료
학습 상태 로드 중...
✓ 학습 상태 로드 완료
✅ 체크포인트 trained_llava-med/emergency_checkpoint_0에서 학습 재개 성공
학습 중 오류 발생: Expected all tensors to be on the same device, but found at least two devices, cuda:3 and cpu!
In DataCollator - Crop image shape: torch.Size([3, 224, 224]), Full image shape: torch.

Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.


2. Saving tokenizer...
3. Merging PEFT model...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some weights of the model checkpoint at microsoft/llava-med-v1.5-mistral-7b were not used when initializing LlavaMistralForCausalLM: {'model.vision_tower.vision_tower.vision_model.encoder.layers.16.mlp.fc2.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.20.self_attn.out_proj.bias', 'model.vision_tower.vision_tower.vision_model.encoder.layers.15.self_attn.q_proj.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.3.layer_norm1.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.20.layer_norm1.bias', 'model.vision_tower.vision_tower.vision_model.encoder.layers.1.self_attn.k_proj.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.1.layer_norm2.bias', 'model.vision_tower.vision_tower.vision_model.encoder.layers.1.self_attn.out_proj.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.20.self_attn.k_proj.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.7.self_attn.v_proj.bias',

4. Saving final merged model...


In [15]:

trainer.save_model('/mnt/engineers/고효진/FNF/saved_model/BiomedCLIP_trained_model')
base_model = AutoModelForCausalLM.from_pretrained('microsoft/llava-med-v1.5-mistral-7b')

trained_model = PeftModel.from_pretrained(base_model, '/mnt/engineers/고효진/FNF/saved_model/BiomedCLIP_trained_model')
merged_trained_model = trained_model.merge_and_unload()
merged_trained_model.save_pretrained('/mnt/engineers/고효진/FNF/saved_model/BiomedCLIP_trained_model', push_to_hub=True, repo_id="sarahyo941/llava-med-v1.5-mistral-7b-oo")

tokenizer.save_pretrained('/mnt/engineers/고효진/FNF/saved_model/merged_trained_model', push_to_hub=True, repo_id='sarahyo941/llava-med-v1.5-mistral-7b-oo')
     

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some weights of the model checkpoint at microsoft/llava-med-v1.5-mistral-7b were not used when initializing LlavaMistralForCausalLM: {'model.vision_tower.vision_tower.vision_model.encoder.layers.22.self_attn.out_proj.bias', 'model.vision_tower.vision_tower.vision_model.encoder.layers.12.layer_norm2.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.15.self_attn.q_proj.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.6.self_attn.q_proj.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.4.self_attn.k_proj.bias', 'model.vision_tower.vision_tower.vision_model.encoder.layers.8.self_attn.v_proj.bias', 'model.vision_tower.vision_tower.vision_model.encoder.layers.19.self_attn.k_proj.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.8.self_attn.v_proj.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.10.self_attn.out_proj.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.9.laye

model-00001-of-00006.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

Upload 6 LFS files:   0%|          | 0/6 [00:00<?, ?it/s]

model-00005-of-00006.safetensors:   0%|          | 0.00/4.83G [00:00<?, ?B/s]

...

model-00004-of-00006.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00002-of-00006.safetensors:   0%|          | 0.00/4.90G [00:00<?, ?B/s]

model-00003-of-00006.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

('/mnt/engineers/고효진/FNF/saved_model/merged_trained_model/tokenizer_config.json',
 '/mnt/engineers/고효진/FNF/saved_model/merged_trained_model/special_tokens_map.json',
 '/mnt/engineers/고효진/FNF/saved_model/merged_trained_model/tokenizer.model',
 '/mnt/engineers/고효진/FNF/saved_model/merged_trained_model/added_tokens.json')